# Laboratorio: Autofabricación y Olas Tecnológicas
### Companion a Vilar (2026b) — Modelo de Logística Anidada

**Paper de referencia:**  
Vilar, J.A. (2026b). *Autofabricación y Olas Tecnológicas: El Modelo de Logística Anidada para la Era de los Robots que se Autorreplican.*  
Working Paper. Universitat Oberta de Catalunya.

**Paper fundacional:**  
Vilar, J.A. (2026a). *Escasez y Resiliencia: Un Marco Matemático para Invertir en la Era de la Autorreplicación Artificial.*  
SSRN Working Paper.

---

## Índice

1. [Setup — importaciones y parámetros base](#1)
2. [Φ_nested(t) — las cuatro olas tecnológicas](#2)
3. [Comparación: modelo simple (Paper I) vs anidado (Paper II)](#3)
4. [Dinámica de feedback robótico — R(t)](#4)
5. [Colapso del coste de producción — C(t)](#5)
6. [Prima de escasez por categoría](#6)
7. [Proyecciones V(t) — portafolio con modelo anidado](#7)
8. [Análisis de sensibilidad — K, γ, t₀](#8)
9. [Detector SOX/Cobre como estimador empírico de t₀](#9)
10. [Conclusiones y referencias](#10)

---
## §1 · Setup <a id='1'></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# --- Estilo oscuro unificado con el resto de laboratorios ---
plt.style.use('dark_background')
COLORS = {
    'ola1': '#00B4D8',   # cian    — LLM / Software IA
    'ola2': '#90E0EF',   # cian claro — Robótica + autofab
    'ola3': '#F77F00',   # naranja — Agentes autónomos
    'ola4': '#F4A261',   # naranja claro — Autofab 2ª fase
    'nested': '#FFFFFF', # blanco  — Φ_nested total
    'simple': '#7B2FBE', # violeta — modelo Paper I
    'conservador': '#48CAE4',
    'base': '#FFFFFF',
    'agresivo': '#F4A261',
    'btc': '#F7931A',
    'cobre': '#CD7F32',
    'uranio': '#39FF14',
    'grid': '#7B2FBE',
    'rv': '#888888',
}
FIGSIZE = (14, 5)
DPI = 110

print("Setup completo.")

In [ ]:
# ============================================================
# PARÁMETROS DEL PAPER II — Tabla 1
# t=0 → enero 2026
# ============================================================

OLAS = [
    # (nombre,           t0,   K,    gamma, color)
    ('Ola 1: LLM/Software IA',        0.5,  1.5,  1.4,  COLORS['ola1']),
    ('Ola 2: Robótica + autofab',      1.0,  2.5,  1.2,  COLORS['ola2']),
    ('Ola 3: Agentes autónomos',       3.0,  3.5,  0.9,  COLORS['ola3']),
    ('Ola 4: Autofab 2ª fase',         5.0,  5.0,  0.7,  COLORS['ola4']),
]

# Parámetros de crecimiento r (SOX proxy)
R_BASE = 0.20
R_ACELERADO = 0.25
R_OPTIMO = 0.30

# Parámetros Paper I (modelo simple) — escenario base
K_SIMPLE  = 2.0
G_SIMPLE  = 0.30
T0_SIMPLE = 3.0

# Capital inicial
CAPITAL = 100_000  # €

# Horizonte temporal
T_MAX = 13   # años (hasta 2039)
t = np.linspace(0, T_MAX, 500)
YEARS = [f"{2026 + int(x)}" for x in range(T_MAX + 1)]

# ============================================================
# FUNCIONES NÚCLEO
# ============================================================

def logistic(t, K, gamma, t0):
    """Curva logística individual: K / (1 + e^(-γ(t-t₀)))"""
    return K / (1 + np.exp(-gamma * (t - t0)))

def phi_nested(t, olas=OLAS):
    """Factor anidado: 1 + Σᵢ Kᵢ/(1+e^(-γᵢ(t-t₀ᵢ)))  — Vilar 2026b, Ec. (1)"""
    return 1 + sum(logistic(t, K, g, t0) for _, t0, K, g, _ in olas)

def phi_simple(t, K=K_SIMPLE, gamma=G_SIMPLE, t0=T0_SIMPLE):
    """Factor simple del Paper I: 1 + K/(1+e^(-γ(t-t₀)))"""
    return 1 + logistic(t, K, gamma, t0)

def V_model(t, Capital, r, phi_func, **phi_kwargs):
    """V(t) = Capital × (1+r)^t × Φ(t)/Φ(0)  — normalizado"""
    phi = phi_func(t, **phi_kwargs)
    phi0 = phi_func(np.array([0.0]), **phi_kwargs)[0]
    return Capital * (1 + r)**t * phi / phi0

# Verificación rápida
phi0 = phi_nested(np.array([0.0]))[0]
phi_ceiling = 1 + sum(K for _, _, K, _, _ in OLAS)
print(f"Φ_nested(0)  = {phi0:.4f}")
print(f"Φ_nested(∞)  ≈ {phi_ceiling:.1f}×  (techo total)")
print(f"Multiplicador neto máximo: {phi_ceiling/phi0:.2f}×")

---
## §2 · Φ_nested(t) — Las cuatro olas tecnológicas <a id='2'></a>

$$\Phi_{\text{nested}}(t) = 1 + \sum_{i=1}^{4} \frac{K_i}{1 + e^{-\gamma_i(t - t_{0i})}}$$

Cada ola $i$ tiene tres parámetros independientes: $K_i$ (amplitud), $\gamma_i$ (velocidad), $t_{0i}$ (año de inflexión). El techo total es $1 + \sum K_i = 13.5\times$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Panel izquierdo: olas individuales + nested ---
ax1 = axes[0]
cumulative = np.ones_like(t)  # base = 1

for nombre, t0, K, gamma, color in OLAS:
    ola = logistic(t, K, gamma, t0)
    ax1.plot(t + 2026, ola, color=color, linewidth=1.8, linestyle='--', label=f"{nombre}  (K={K}, γ={gamma}, t₀={int(t0+2026)})")
    cumulative = cumulative + ola

# Φ_nested total
phi_tot = phi_nested(t)
ax1.plot(t + 2026, phi_tot, color=COLORS['nested'], linewidth=2.5, label=f'Φ_nested(t) total  (techo={phi_ceiling}×)')

# Línea de techo
ax1.axhline(phi_ceiling, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)
ax1.text(2038, phi_ceiling + 0.1, f'Techo = {phi_ceiling}×', color='gray', fontsize=8)

ax1.set_title('Olas tecnológicas — contribuciones individuales y Φ_nested(t)', fontsize=11)
ax1.set_xlabel('Año')
ax1.set_ylabel('Contribución a Φ_nested')
ax1.legend(fontsize=7.5, loc='upper left')
ax1.set_xlim(2026, 2039)
ax1.grid(alpha=0.2)

# --- Panel derecho: velocidad de adopción (derivada de Φ_nested) ---
ax2 = axes[1]
dphi = np.gradient(phi_tot, t)
ax2.fill_between(t + 2026, dphi, alpha=0.25, color=COLORS['nested'])
ax2.plot(t + 2026, dphi, color=COLORS['nested'], linewidth=2)

# Marcar t₀ de cada ola
for nombre, t0, K, gamma, color in OLAS:
    ax2.axvline(t0 + 2026, color=color, linewidth=1.0, linestyle='--', alpha=0.7)
    ax2.text(t0 + 2026 + 0.05, ax2.get_ylim()[1] * 0.85 if ax2.get_ylim()[1] != 0 else 1,
             f"t₀={int(t0+2026)}", color=color, fontsize=7.5, rotation=90)

ax2.set_title('Velocidad de adopción dΦ/dt — picos de máxima aceleración', fontsize=11)
ax2.set_xlabel('Año')
ax2.set_ylabel('dΦ_nested / dt  (tasa de cambio anual)')
ax2.set_xlim(2026, 2039)
ax2.grid(alpha=0.2)

plt.suptitle('Modelo de Logística Anidada — Vilar (2026b)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## §3 · Paper I vs Paper II — comparación de modelos <a id='3'></a>

El Paper I usa $\Phi_L(t)$ con un único conjunto $(K, \gamma, t_0)$. El Paper II reemplaza esa curva por $\Phi_{\text{nested}}(t)$, suma de cuatro logísticas. Ambos modelos convergen en $V(0) = \text{Capital}$ por la normalización $\Phi(t)/\Phi(0)$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Panel izquierdo: Φ simple vs Φ nested ---
ax1 = axes[0]

# Paper I — tres escenarios (K=2,4,6)
for K_s, gamma_s, t0_s, ls, label in [
    (2.0, 0.30, 3.0, '-',  'Paper I · base  (K=2, γ=0.30)'),
    (4.0, 0.40, 3.0, '--', 'Paper I · acelerado  (K=4, γ=0.40)'),
    (6.0, 0.50, 3.0, ':',  'Paper I · óptimo  (K=6, γ=0.50)'),
]:
    phi_s = phi_simple(t, K=K_s, gamma=gamma_s, t0=t0_s)
    phi_s_norm = phi_s / phi_s[0]
    ax1.plot(t + 2026, phi_s_norm, color=COLORS['simple'], linewidth=1.8, linestyle=ls, alpha=0.8, label=label)

# Paper II — nested normalizado
phi_n = phi_nested(t)
phi_n_norm = phi_n / phi_n[0]
ax1.plot(t + 2026, phi_n_norm, color=COLORS['nested'], linewidth=2.5, label=f'Paper II · nested  (techo ~{phi_ceiling/phi_n[0]:.1f}× neto)')

ax1.set_title('Φ normalizado: Paper I (logística simple) vs Paper II (anidada)', fontsize=10)
ax1.set_xlabel('Año')
ax1.set_ylabel('Φ(t) / Φ(0)')
ax1.legend(fontsize=8)
ax1.set_xlim(2026, 2039)
ax1.grid(alpha=0.2)

# --- Panel derecho: V(t) en € ---
ax2 = axes[1]

# Paper I escenarios
for K_s, gamma_s, r_s, ls, label, alpha in [
    (2.0, 0.30, R_BASE,      '-',  'P-I base',        0.6),
    (4.0, 0.40, R_ACELERADO, '--', 'P-I acelerado',   0.6),
    (6.0, 0.50, R_OPTIMO,    ':',  'P-I óptimo',      0.6),
]:
    V_s = V_model(t, CAPITAL, r_s, phi_simple, K=K_s, gamma=gamma_s, t0=T0_SIMPLE)
    ax2.plot(t + 2026, V_s / 1000, color=COLORS['simple'], linewidth=1.8, linestyle=ls, alpha=alpha, label=label)

# Paper II — nested con r=base
V_n = V_model(t, CAPITAL, R_BASE, phi_nested)
ax2.plot(t + 2026, V_n / 1000, color=COLORS['nested'], linewidth=2.5, label='P-II nested (r=20%)')

# Compound puro (sin Φ)
V_pure = CAPITAL * (1 + R_BASE)**t
ax2.plot(t + 2026, V_pure / 1000, color='gray', linewidth=1.2, linestyle='--', alpha=0.5, label='(1+r)^t puro  (sin Φ)')

ax2.set_title(f'V(t) en k€ — Capital inicial {CAPITAL/1000:.0f}k€', fontsize=10)
ax2.set_xlabel('Año')
ax2.set_ylabel('Valor del portfolio (k€)')
ax2.legend(fontsize=8)
ax2.set_xlim(2026, 2039)
ax2.grid(alpha=0.2)

plt.suptitle('Comparación Paper I vs Paper II — V(t) con capital base 100k€', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Tabla de valores
print("\n=== Valores V(t) en k€ al final del horizonte (año 2039, t=13) ===")
t_final = T_MAX
for K_s, gamma_s, r_s, label in [
    (2.0, 0.30, R_BASE,      'Paper I base'),
    (4.0, 0.40, R_ACELERADO, 'Paper I acelerado'),
    (6.0, 0.50, R_OPTIMO,    'Paper I óptimo'),
]:
    V_end = V_model(np.array([t_final]), CAPITAL, r_s, phi_simple, K=K_s, gamma=gamma_s, t0=T0_SIMPLE)[0]
    print(f"  {label:<22}: {V_end/1000:>8.1f} k€  ({V_end/CAPITAL:.1f}×)")

V_nested_end = V_model(np.array([t_final]), CAPITAL, R_BASE, phi_nested)[0]
print(f"  {'Paper II nested':<22}: {V_nested_end/1000:>8.1f} k€  ({V_nested_end/CAPITAL:.1f}×)")

V_pure_end = CAPITAL * (1 + R_BASE)**t_final
print(f"  {'Compound puro':<22}: {V_pure_end/1000:>8.1f} k€  ({V_pure_end/CAPITAL:.1f}×)")

---
## §4 · Dinámica de feedback robótico — R(t) <a id='4'></a>

Cuando los robots pueden fabricar robots, la capacidad productiva sigue:

$$\frac{dR}{dt} = \alpha_R \cdot \eta(t) \cdot R(t) + P_H$$

Con solución analítica (η constante):

$$R(t) = \left(R_0 + \frac{P_H}{\alpha_R}\right) e^{\alpha_R \eta t} - \frac{P_H}{\alpha_R}$$

**Parámetros** (Tabla 2, paper II): $\alpha_R = 0.30$, $P_H = 0.05$, $R_0 = 1.0$.

In [ ]:
# --- Parámetros del modelo de feedback ---
# α_R derivado del escenario base (Tabla 2): R(3)=3.0×, η=0.20
# → α_R·η·t = ln(R) → α_R = ln(3)/(0.20×3) ≈ 1.83
ALPHA_R = 1.83   # tasa de replicación (calibrada en Tabla 2, paper II)
P_H     = 0.05   # componente humana exógena
R0      = 1.0    # capacidad inicial normalizada

ETA_SCENARIOS = [
    (0.05, 'Conservador (η=0.05)', COLORS['conservador']),
    (0.20, 'Base (η=0.20)',         COLORS['base']),
    (0.50, 'Agresivo (η=0.50)',     COLORS['agresivo']),
]

# La autofabricación empieza en t₀ = 1.0 (2027, Ola 2)
T_AUTOFAB_START = 1.0  # años desde 2026

def robot_capacity(t_rel, eta, alpha_R=ALPHA_R, P_H=P_H, R0=R0):
    """R(t) = (R₀ + P_H/α_R)·e^(α_R·η·t) − P_H/α_R"""
    A = R0 + P_H / alpha_R
    return A * np.exp(alpha_R * eta * t_rel) - P_H / alpha_R

t_robot = np.linspace(0, 10, 300)  # años desde inicio de autofabricación (2027)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Panel izquierdo: R(t) ---
ax1 = axes[0]
for eta, label, color in ETA_SCENARIOS:
    R = robot_capacity(t_robot, eta)
    ax1.plot(t_robot + 2027, R, color=color, linewidth=2.2, label=label)

ax1.set_title('Capacidad robótica R(t) — desde 2027 (inicio ola 2)', fontsize=10)
ax1.set_xlabel('Año')
ax1.set_ylabel('R(t)  [normalizado, R(2027)=1]')
ax1.legend(fontsize=9)
ax1.set_xlim(2027, 2037)
ax1.grid(alpha=0.2)

# Marcar hitos de capacidad
for mult in [2, 5, 10, 20]:
    ax1.axhline(mult, color='gray', linewidth=0.5, linestyle=':', alpha=0.4)
    ax1.text(2027.1, mult + 0.3, f'{mult}×', color='gray', fontsize=7)

# --- Panel derecho: coste relativo C(t) = 1/R(t) ---
ax2 = axes[1]
for eta, label, color in ETA_SCENARIOS:
    R = robot_capacity(t_robot, eta)
    C = 1.0 / R
    ax2.plot(t_robot + 2027, C * 100, color=color, linewidth=2.2, label=label)

ax2.set_title('Coste de producción C(t) ∝ 1/R(t)  [% relativo a 2027]', fontsize=10)
ax2.set_xlabel('Año')
ax2.set_ylabel('Coste relativo (%)')
ax2.legend(fontsize=9)
ax2.set_xlim(2027, 2037)
ax2.set_ylim(0, 105)
ax2.grid(alpha=0.2)

# Línea de referencia
for pct in [50, 25, 10]:
    ax2.axhline(pct, color='gray', linewidth=0.5, linestyle=':', alpha=0.4)
    ax2.text(2027.1, pct + 1, f'{pct}%', color='gray', fontsize=7)

plt.suptitle('Feedback robótico — dR/dt = α_R · η · R(t) + P_H  (Vilar 2026b, §3)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Tabla resumen
print("\n=== Proyecciones de capacidad y coste relativo (Tabla 2, paper II) ===")
print(f"{'η Escenario':<26} | {'2030 (t+3)':>10} | {'2032 (t+5)':>10} | {'2035 (t+8)':>10} | {'Coste 2032':>10}")
print('-' * 76)
for eta, label, _ in ETA_SCENARIOS:
    r3 = robot_capacity(3, eta);  r5 = robot_capacity(5, eta);  r8 = robot_capacity(8, eta)
    cost5 = 1/r5 * 100
    print(f"{label:<26} | {r3:>8.1f}× | {r5:>8.1f}× | {r8:>8.1f}× | {cost5:>8.1f}%")

---
## §5 · Colapso del coste de producción <a id='5'></a>

La velocidad de caída del coste es la variable económica clave. En el escenario base (η=0.20), el coste unitario se reduce a la mitad en ~3.5 años desde el inicio de la autofabricación.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Panel izquierdo: tiempo de vida media del coste ---
ax1 = axes[0]

eta_range = np.linspace(0.01, 0.6, 200)
half_life = []
for eta in eta_range:
    # Encontrar t donde C(t) = 0.50 (coste a la mitad)
    t_hl = np.linspace(0, 20, 5000)
    R_hl = robot_capacity(t_hl, eta)
    C_hl = 1.0 / R_hl
    idx = np.argmin(np.abs(C_hl - 0.50))
    half_life.append(t_hl[idx])

ax1.plot(eta_range, half_life, color=COLORS['base'], linewidth=2)
ax1.fill_between(eta_range, half_life, alpha=0.15, color=COLORS['base'])

# Marcar escenarios del paper
for eta, label, color in ETA_SCENARIOS:
    t_hl_val = np.interp(eta, eta_range, half_life)
    ax1.scatter([eta], [t_hl_val], color=color, s=80, zorder=5)
    ax1.annotate(f"{label.split('(')[0].strip()}\n{t_hl_val:.1f} años",
                 xy=(eta, t_hl_val), xytext=(eta + 0.02, t_hl_val + 0.5),
                 fontsize=8, color=color,
                 arrowprops=dict(arrowstyle='->', color=color, lw=0.8))

ax1.set_title('Semivida del coste de producción — t(C=50%) vs η', fontsize=10)
ax1.set_xlabel('Eficiencia de autorreplicación η')
ax1.set_ylabel('Años para reducir coste al 50%')
ax1.grid(alpha=0.2)

# Referencia histórica
ax1.axhline(20, color='orange', linewidth=0.8, linestyle='--', alpha=0.6)
ax1.text(0.45, 20.5, 'Revolución textil Lancashire (~20 años)', color='orange', fontsize=7)
ax1.axhline(2, color='#90E0EF', linewidth=0.8, linestyle='--', alpha=0.6)
ax1.text(0.45, 2.5, 'Ley de Moore (~2 años/ciclo)', color='#90E0EF', fontsize=7)

# --- Panel derecho: coste por sector en 2032 ---
ax2 = axes[1]

SECTORES = [
    ('Manufactura estándar', 0.90, '#FF6B6B'),
    ('Logística/distribución', 0.70, '#FF9F43'),
    ('Construcción',           0.65, '#FECA57'),
    ('Software / AI',          0.45, '#48DBFB'),
    ('Energía nuclear',        0.20, COLORS['uranio']),
    ('Metales críticos (Cu)',  0.10, COLORS['cobre']),
    ('Grid / infraestructura', 0.20, COLORS['grid']),
    ('Bitcoin',                0.00, COLORS['btc']),
]

# En 2032 (t=5 desde 2027), escenario base
R_2032 = robot_capacity(5, 0.20)  # η=0.20 base
C_2032_base = 1.0 / R_2032  # coste relativo de manufactura pura

nombres = [s[0] for s in SECTORES]
rho = [s[1] for s in SECTORES]
colores = [s[2] for s in SECTORES]

# Coste efectivo del sector = ρ·C_manufactura + (1-ρ)·1.0 (la parte escasa mantiene precio)
costes_2032 = [r * C_2032_base + (1 - r) * 1.0 for r in rho]

bars = ax2.barh(nombres, [c * 100 for c in costes_2032], color=colores, alpha=0.85)
ax2.axvline(100, color='white', linewidth=0.8, linestyle='--', alpha=0.4)
ax2.axvline(C_2032_base * 100, color=COLORS['agresivo'], linewidth=1.2, linestyle=':',
            label=f'Manufactura pura 100% replicable = {C_2032_base*100:.0f}%')

for bar, cost in zip(bars, costes_2032):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{cost*100:.0f}%', va='center', fontsize=8)

ax2.set_title(f'Coste relativo por sector en 2032 — escenario base (η=0.20)', fontsize=10)
ax2.set_xlabel('Coste relativo (100% = nivel 2027)')
ax2.legend(fontsize=8)
ax2.set_xlim(0, 115)
ax2.grid(alpha=0.2, axis='x')

plt.suptitle('Colapso del coste de producción — implicaciones sectoriales (2032)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## §6 · Prima de escasez por categoría de activo <a id='6'></a>

$$S_i(t) = 1 + \varepsilon_i \cdot \frac{R(t) - 1}{R(t)}$$

donde $\varepsilon_i \in [0,1]$ es el coeficiente de escasez del activo. Cuando $R \to \infty$: $S_i \to 1 + \varepsilon_i$.

In [ ]:
# Tabla 3 del paper II
ACTIVOS_ESCASEZ = [
    ('Bitcoin',            1.00, COLORS['btc']),
    ('Uranio',             0.75, COLORS['uranio']),
    ('Cobre',              0.70, COLORS['cobre']),
    ('Energía / Grid',     0.65, COLORS['grid']),
    ('Tecnología / AI',    0.45, '#00B4D8'),
    ('Renta Variable',     0.20, COLORS['rv']),
]

def scarcity_premium(t_rel, epsilon, eta=0.20):
    """S_i(t) = 1 + ε_i · (R(t)-1)/R(t)  — Vilar 2026b, Ec. (5)"""
    R = robot_capacity(t_rel, eta)
    return 1 + epsilon * (R - 1) / R

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Panel izquierdo: S_i(t) en el tiempo ---
ax1 = axes[0]
t_scarcity = np.linspace(0, 10, 300)

for nombre, eps, color in ACTIVOS_ESCASEZ:
    S = scarcity_premium(t_scarcity, eps)
    ax1.plot(t_scarcity + 2027, S, color=color, linewidth=2.2, label=f'{nombre}  (ε={eps:.2f}, lím={1+eps:.2f}×)')

ax1.set_title('Prima de escasez S_i(t) por categoría — escenario base (η=0.20)', fontsize=10)
ax1.set_xlabel('Año')
ax1.set_ylabel('S_i(t)  [1.0 = sin prima]')
ax1.legend(fontsize=8.5)
ax1.set_xlim(2027, 2037)
ax1.grid(alpha=0.2)
ax1.axhline(1.0, color='gray', linewidth=0.6, linestyle='--', alpha=0.5)

# --- Panel derecho: prima en 2032 por escenario η ---
ax2 = axes[1]

nombres = [a[0] for a in ACTIVOS_ESCASEZ]
eps_vals = [a[1] for a in ACTIVOS_ESCASEZ]
colores  = [a[2] for a in ACTIVOS_ESCASEZ]
x = np.arange(len(nombres))
width = 0.25

for i, (eta, label, color) in enumerate(ETA_SCENARIOS):
    S_2032 = [scarcity_premium(5, eps, eta=eta) for eps in eps_vals]
    ax2.bar(x + i * width, S_2032, width, label=label, color=color, alpha=0.80)

ax2.set_title('Prima de escasez S_i en 2032 — por escenario η', fontsize=10)
ax2.set_xlabel('Categoría de activo')
ax2.set_ylabel('S_i  [1.0 = sin prima]')
ax2.set_xticks(x + width)
ax2.set_xticklabels(nombres, rotation=20, ha='right', fontsize=8)
ax2.legend(fontsize=8)
ax2.axhline(1.0, color='gray', linewidth=0.6, linestyle='--', alpha=0.5)
ax2.grid(alpha=0.2, axis='y')

plt.suptitle('Prima de escasez — activos con oferta inelástica ganan en términos relativos', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Tabla de coeficientes
print("\n=== Tabla 3 — Coeficientes de escasez y prima límite (Vilar 2026b) ===")
print(f"{'Categoría':<25} | {'ε':>5} | {'S_i(∞)':>8} | {'S_i 2032 base':>14} | {'Justificación'}")
print('-' * 90)
JUSTIF = {
    'Bitcoin': 'Oferta fija por protocolo; imposible de replicar',
    'Uranio': 'Escasez geológica + ciclo minero 10-15 años',
    'Cobre': 'Recurso geológico; reciclaje parcial pero oferta inelástica',
    'Energía / Grid': 'Infraestructura física; escala pero no se imprime',
    'Tecnología / AI': 'Software replicable, hardware + talento con fricción',
    'Renta Variable': 'Beta mercado; mix de replicables y escasos',
}
for nombre, eps, _ in ACTIVOS_ESCASEZ:
    s_lim = 1 + eps
    s_2032 = scarcity_premium(5, eps, eta=0.20)
    print(f"{nombre:<25} | {eps:>5.2f} | {s_lim:>8.2f}× | {s_2032:>12.3f}× | {JUSTIF.get(nombre, '')}")

---
## §7 · Proyecciones V(t) — portfolio con modelo anidado <a id='7'></a>

In [ ]:
# Pesos aproximados de la cartera Escasez y Resiliencia
# (orden: Bitcoin, Cobre+Metales, Uranio, Grid, AI/Tech, RentaVariable)
PESOS_CARTERA = {
    'Bitcoin':         0.25,
    'Metales':         0.12,
    'Uranio':          0.10,
    'Grid':            0.08,
    'AI / Tech':       0.25,
    'Renta Variable':  0.20,
}

EPS_CARTERA = {
    'Bitcoin':        1.00,
    'Metales':        0.70,
    'Uranio':         0.75,
    'Grid':           0.65,
    'AI / Tech':      0.45,
    'Renta Variable': 0.20,
}

COLORES_CARTERA = {
    'Bitcoin':        COLORS['btc'],
    'Metales':        COLORS['cobre'],
    'Uranio':         COLORS['uranio'],
    'Grid':           COLORS['grid'],
    'AI / Tech':      '#00B4D8',
    'Renta Variable': COLORS['rv'],
}

def V_cartera_nested(t_arr, Capital, r, eta_robot=0.20):
    """Portfolio ponderado: cada pilar ajustado por su prima de escasez"""
    V_total = np.zeros_like(t_arr, dtype=float)
    t_robot_arr = np.maximum(t_arr - T_AUTOFAB_START, 0)
    
    for pilar, peso in PESOS_CARTERA.items():
        eps = EPS_CARTERA[pilar]
        # Φ_nested proporciona el multiplicador tecnológico general
        phi = phi_nested(t_arr)
        phi0 = phi_nested(np.array([0.0]))[0]
        # Prima de escasez aplicada al pilar
        S = scarcity_premium(t_robot_arr, eps, eta=eta_robot)
        # Valor del pilar
        V_pilar = Capital * peso * (1 + r)**t_arr * (phi / phi0) * S
        V_total += V_pilar
    return V_total

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Panel izquierdo: V(t) por escenario ---
ax1 = axes[0]

ESCENARIOS_V = [
    (R_BASE,      0.05, COLORS['conservador'], '-',  'Conservador  (r=20%, η=0.05)'),
    (R_BASE,      0.20, COLORS['base'],        '-',  'Base         (r=20%, η=0.20)'),
    (R_ACELERADO, 0.20, COLORS['agresivo'],    '--', 'Acelerado    (r=25%, η=0.20)'),
    (R_OPTIMO,    0.50, '#FF6B6B',             ':',  'Óptimo       (r=30%, η=0.50)'),
]

for r_, eta_, color, ls, label in ESCENARIOS_V:
    V_ = V_cartera_nested(t, CAPITAL, r_, eta_robot=eta_)
    ax1.plot(t + 2026, V_ / 1000, color=color, linewidth=2.2, linestyle=ls, label=label)

# Compound puro de referencia
ax1.plot(t + 2026, CAPITAL * (1 + R_BASE)**t / 1000,
         color='gray', linewidth=1.0, linestyle='--', alpha=0.5, label='Compound puro (r=20%)')

ax1.set_title(f'V(t) portfolio completo — Capital inicial {CAPITAL/1000:.0f}k€', fontsize=10)
ax1.set_xlabel('Año')
ax1.set_ylabel('Valor del portfolio (k€)')
ax1.legend(fontsize=8)
ax1.set_xlim(2026, 2039)
ax1.grid(alpha=0.2)

# --- Panel derecho: composición por pilar en 2032 (escenario base) ---
ax2 = axes[1]

t_2032 = 6.0  # 2032 = 2026 + 6
pilares_val = {}
t_rob_2032 = max(t_2032 - T_AUTOFAB_START, 0)
phi_2032 = phi_nested(np.array([t_2032]))[0]
phi0_val = phi_nested(np.array([0.0]))[0]

for pilar, peso in PESOS_CARTERA.items():
    eps = EPS_CARTERA[pilar]
    S_2032 = scarcity_premium(t_rob_2032, eps, eta=0.20)
    V_pilar_2032 = CAPITAL * peso * (1 + R_BASE)**t_2032 * (phi_2032 / phi0_val) * S_2032
    pilares_val[pilar] = V_pilar_2032

total_2032 = sum(pilares_val.values())
labels_pie = [f"{k}\n{v/1000:.0f}k€\n({v/total_2032*100:.0f}%)" for k, v in pilares_val.items()]
colors_pie = [COLORES_CARTERA[k] for k in pilares_val.keys()]
sizes_pie = list(pilares_val.values())

wedges, texts = ax2.pie(sizes_pie, labels=labels_pie, colors=colors_pie,
                         startangle=90, pctdistance=0.85,
                         textprops={'fontsize': 7.5},
                         wedgeprops={'edgecolor': 'black', 'linewidth': 0.5})

ax2.set_title(f'Distribución por pilar en 2032\nTotal: {total_2032/1000:.0f}k€ — escenario base', fontsize=10)

plt.suptitle('Proyecciones portfolio Escasez y Resiliencia — modelo anidado (Paper II)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Tabla final
print(f"\n=== Valores finales por escenario (año 2039, t=13) ===")
print(f"{'Escenario':<40} | {'Valor final':>12} | {'Multiplicador':>14}")
print('-' * 72)
for r_, eta_, _, _, label in ESCENARIOS_V:
    V_end = V_cartera_nested(np.array([13.0]), CAPITAL, r_, eta_robot=eta_)[0]
    print(f"{label:<40} | {V_end/1000:>10.1f}k€ | {V_end/CAPITAL:>12.1f}×")

---
## §8 · Análisis de sensibilidad — K, γ, t₀ por ola <a id='8'></a>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

t_eval = np.array([13.0])  # 2039

def phi_nested_custom(t, olas_custom):
    """Φ_nested con lista de olas customizable"""
    return 1 + sum(logistic(t, K, g, t0) for _, t0, K, g, _ in olas_custom)

# --- Panel 1: sensibilidad a K de la ola 4 (la más relevante) ---
ax1 = axes[0]
K4_range = np.linspace(1.0, 10.0, 50)

for ola_idx, (ola_nombre, _, _, _, color) in enumerate(OLAS):
    V_vals = []
    for K_var in K4_range:
        olas_mod = [
            (n, t0, K_var if i == ola_idx else K, g, c)
            for i, (n, t0, K, g, c) in enumerate(OLAS)
        ]
        phi_t = phi_nested_custom(t_eval, olas_mod)
        phi0  = phi_nested_custom(np.array([0.0]), olas_mod)
        V_ = CAPITAL * (1 + R_BASE)**13 * phi_t / phi0
        V_vals.append(V_[0] / 1000)
    ax1.plot(K4_range, V_vals, color=color, linewidth=2, label=ola_nombre.split(':')[1].strip())

ax1.axvline(OLAS[3][2], color='gray', linewidth=0.8, linestyle='--', alpha=0.6, label='K_ola4 calibrado')
ax1.set_title('Sensibilidad a K por ola — V(2039) en k€', fontsize=10)
ax1.set_xlabel('K (amplitud de la ola variada)')
ax1.set_ylabel('V(2039) (k€)')
ax1.legend(fontsize=7.5)
ax1.grid(alpha=0.2)

# --- Panel 2: sensibilidad a γ de la ola 4 ---
ax2 = axes[1]
gamma4_range = np.linspace(0.1, 2.5, 50)

for ola_idx, (ola_nombre, _, _, _, color) in enumerate(OLAS):
    V_vals = []
    for g_var in gamma4_range:
        olas_mod = [
            (n, t0, K, g_var if i == ola_idx else g, c)
            for i, (n, t0, K, g, c) in enumerate(OLAS)
        ]
        phi_t = phi_nested_custom(t_eval, olas_mod)
        phi0  = phi_nested_custom(np.array([0.0]), olas_mod)
        V_ = CAPITAL * (1 + R_BASE)**13 * phi_t / phi0
        V_vals.append(V_[0] / 1000)
    ax2.plot(gamma4_range, V_vals, color=color, linewidth=2, label=ola_nombre.split(':')[1].strip())

ax2.set_title('Sensibilidad a γ por ola — V(2039) en k€', fontsize=10)
ax2.set_xlabel('γ (velocidad de adopción de la ola variada)')
ax2.set_ylabel('V(2039) (k€)')
ax2.legend(fontsize=7.5)
ax2.grid(alpha=0.2)

# --- Panel 3: sensibilidad a t₀ de la ola 4 ---
ax3 = axes[2]
t0_range = np.linspace(1.0, 10.0, 50)  # años desde 2026

for ola_idx, (ola_nombre, _, _, _, color) in enumerate(OLAS):
    V_vals = []
    for t0_var in t0_range:
        olas_mod = [
            (n, t0_var if i == ola_idx else t0, K, g, c)
            for i, (n, t0, K, g, c) in enumerate(OLAS)
        ]
        phi_t = phi_nested_custom(t_eval, olas_mod)
        phi0  = phi_nested_custom(np.array([0.0]), olas_mod)
        V_ = CAPITAL * (1 + R_BASE)**13 * phi_t / phi0
        V_vals.append(V_[0] / 1000)
    ax3.plot(t0_range + 2026, V_vals, color=color, linewidth=2, label=ola_nombre.split(':')[1].strip())

ax3.set_title('Sensibilidad a t₀ por ola — V(2039) en k€', fontsize=10)
ax3.set_xlabel('t₀ de la ola variada (año de inflexión)')
ax3.set_ylabel('V(2039) (k€)')
ax3.legend(fontsize=7.5)
ax3.grid(alpha=0.2)

plt.suptitle('Análisis de sensibilidad — parámetros K, γ, t₀ de cada ola (V en 2039)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## §9 · SOX/Cobre como detector de t₀ <a id='9'></a>

El detector SOX/Cobre del paper fundacional identifica el momento empírico en que la IA empieza a **miniaturizar o sustituir** el input físico. Ese cruce corresponde al $t_0$ estimado del modelo.

Aquí simulamos cuatro posibles fechas de $t_0$ (2026–2029) y su impacto sobre V(t).

In [ ]:
# Estados SOX/Cobre
ESTADOS_SOX = [
    ('ERA CONSTRUCCIÓN\n(SOX↑ + Cu↑)',  '#00B4D8', 'both assets rising'),
    ('DIVERGENCIA·ALERTA\n(SOX↑, Cu↓)', '#FF6B6B', 'IA starts substituting physical'),
    ('ESCASEZ FÍSICA\n(SOX↓, Cu↑)',     '#FECA57', 'commodities lead'),
    ('CONTRACCIÓN\n(SOX↓ + Cu↓)',       '#888888', 'defensive mode'),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Panel izquierdo: impacto de t₀ global sobre V(2039) ---
ax1 = axes[0]

# Simular t₀ como el año de inflexión de la ola 2 (robot autofab)
t0_2_range = np.linspace(0.5, 6.0, 50)  # años desde 2026

V_t0_impact = []
for t0_2 in t0_2_range:
    olas_sim = [
        (OLAS[0][0], OLAS[0][1], OLAS[0][2], OLAS[0][3], OLAS[0][4]),
        (OLAS[1][0], t0_2,       OLAS[1][2], OLAS[1][3], OLAS[1][4]),
        (OLAS[2][0], t0_2 + 2.0, OLAS[2][2], OLAS[2][3], OLAS[2][4]),
        (OLAS[3][0], t0_2 + 4.0, OLAS[3][2], OLAS[3][3], OLAS[3][4]),
    ]
    phi_t = phi_nested_custom(t_eval, olas_sim)
    phi0  = phi_nested_custom(np.array([0.0]), olas_sim)
    V_ = CAPITAL * (1 + R_BASE)**13 * phi_t / phi0
    V_t0_impact.append(V_[0] / 1000)

ax1.plot(t0_2_range + 2026, V_t0_impact, color=COLORS['base'], linewidth=2.2)
ax1.fill_between(t0_2_range + 2026, V_t0_impact,
                 min(V_t0_impact), alpha=0.15, color=COLORS['base'])

# Marcar calibración actual
t0_calibrado = OLAS[1][1]  # 1.0 → 2027
V_calibrado = V_t0_impact[np.argmin(np.abs(t0_2_range - t0_calibrado))]
ax1.scatter([t0_calibrado + 2026], [V_calibrado], color='#F7931A', s=100, zorder=5,
            label=f't₀_ola2 calibrado (2027) → {V_calibrado:.0f}k€')

# Zona de DIVERGENCIA SOX/Cobre
ax1.axvspan(2026.5, 2028, alpha=0.10, color='#FF6B6B', label='Zona DIVERGENCIA observable')

ax1.set_title('Impacto de t₀ (ola 2) sobre V(2039) — cuánto importa la fecha del cruce', fontsize=10)
ax1.set_xlabel('Año de inflexión ola 2 (t₀_autofab)')
ax1.set_ylabel('V(2039) (k€)')
ax1.legend(fontsize=8)
ax1.grid(alpha=0.2)

# --- Panel derecho: semáforo SOX/Cobre → acción de cartera ---
ax2 = axes[1]
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.axis('off')

titulo = 'DETECTOR SOX/COBRE — Guía de acción'
ax2.text(0.5, 0.95, titulo, ha='center', va='top', fontsize=11, fontweight='bold', color='white')

ACCIONES = [
    ('🟢 ERA DE CONSTRUCCIÓN',  'SOX ↑  y  Cu ↑',
     'Mantener metales + IA\nTesis completa en marcha',    '#00B4D8'),
    ('🔴 DIVERGENCIA · ALERTA', 'SOX ↑  /  Cu ↓  ← t₀ EMPÍRICO',
     'Reducir mineras, reforzar IA pura\nMarcar t₀ en el modelo',  '#FF6B6B'),
    ('🟡 ESCASEZ FÍSICA',       'SOX ↓  /  Cu ↑',
     'Metales fuertes\nEsperar confirmación de rebote SOX', '#FECA57'),
    ('⚫ CONTRACCIÓN',          'SOX ↓  y  Cu ↓',
     'Modo defensivo\nEsperar estabilización',              '#888888'),
]

for i, (estado, condicion, accion, color) in enumerate(ACCIONES):
    y = 0.78 - i * 0.20
    rect = plt.Rectangle((0.02, y - 0.05), 0.96, 0.17,
                          facecolor=color, alpha=0.15, transform=ax2.transAxes)
    ax2.add_patch(rect)
    ax2.text(0.05, y + 0.06, estado, ha='left', va='center', fontsize=9,
             fontweight='bold', color=color, transform=ax2.transAxes)
    ax2.text(0.05, y - 0.00, condicion, ha='left', va='center', fontsize=8,
             color='white', transform=ax2.transAxes)
    ax2.text(0.05, y - 0.07, accion, ha='left', va='center', fontsize=7.5,
             color='#CCCCCC', transform=ax2.transAxes)

plt.suptitle('SOX/Cobre como estimador empírico de t₀ — Vilar (2026a, 2026b)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## §10 · Conclusiones y referencias <a id='10'></a>

In [ ]:
print("=" * 70)
print("RESUMEN — MODELO DE LOGÍSTICA ANIDADA (VILAR, 2026b)")
print("=" * 70)

print("\n1. MODELO PRINCIPAL")
print("   Φ_nested(t) = 1 + Σᵢ Kᵢ/(1+e^(-γᵢ(t-t₀ᵢ)))")
print("   V(t) = Capital × (1+r)^t × Φ_nested(t)/Φ_nested(0)")

print("\n2. CUATRO OLAS CALIBRADAS (Tabla 1)")
print(f"   {'Ola':<35} {'t₀':>5} {'K':>5} {'γ':>5}")
print("   " + "-" * 52)
for nombre, t0, K, gamma, _ in OLAS:
    print(f"   {nombre:<35} {int(t0+2026):>5} {K:>5.1f} {gamma:>5.1f}")
print(f"   {'Techo total (1 + ΣK):':<35} {'':>5} {1+sum(K for _,_,K,_,_ in OLAS):>5.1f}")

print("\n3. ROBOT FEEDBACK (Tabla 2) — inicio en 2027")
print(f"   α_R={ALPHA_R}, P_H={P_H}, R₀={R0}")
print(f"   {'Escenario':<20} {'R(2030)':>8} {'R(2032)':>8} {'C(2032)':>8}")
print("   " + "-" * 48)
for eta, label, _ in ETA_SCENARIOS:
    r3 = robot_capacity(3, eta); r5 = robot_capacity(5, eta)
    print(f"   {label:<20} {r3:>7.1f}× {r5:>7.1f}× {1/r5*100:>7.1f}%")

print("\n4. PRIMA DE ESCASEZ (Tabla 3) — límite R→∞")
print(f"   {'Activo':<20} {'ε':>5} {'S_i(∞)':>8} {'S(2032 base)':>14}")
print("   " + "-" * 52)
for nombre, eps, _ in ACTIVOS_ESCASEZ:
    s_lim = 1 + eps
    s_2032 = scarcity_premium(5, eps, eta=0.20)
    print(f"   {nombre:<20} {eps:>5.2f} {s_lim:>7.2f}× {s_2032:>12.3f}×")

print("\n5. PROYECCIONES V(2039) — Capital inicial 100k€")
for r_, eta_, _, _, label in ESCENARIOS_V:
    V_end = V_cartera_nested(np.array([13.0]), CAPITAL, r_, eta_robot=eta_)[0]
    print(f"   {label:<40} → {V_end/1000:>7.1f}k€  ({V_end/CAPITAL:.1f}×)")

print("\n" + "=" * 70)
print("REFERENCIAS")
print("=" * 70)
print("""
Vilar, J.A. (2026a). Escasez y Resiliencia: Un Marco Matemático para
  Invertir en la Era de la Autorreplicación Artificial.
  SSRN Working Paper. https://papers.ssrn.com/

Vilar, J.A. (2026b). Autofabricación y Olas Tecnológicas: El Modelo de
  Logística Anidada para la Era de los Robots que se Autorreplican.
  Working Paper. Universitat Oberta de Catalunya.

Verhulst, P.-F. (1838). Notice sur la loi que la population suit dans
  son accroissement. Correspondance Mathématique et Physique, 10, 113–121.

Bass, F.M. (1969). A new product growth for model consumer durables.
  Management Science, 15(5), 215–227.

Freeman, C. & Louçã, F. (2001). As Time Goes By: From the Industrial
  Revolutions to the Information Revolution. Oxford University Press.

Mokyr, J. (2002). The Gifts of Athena: Historical Origins of the
  Knowledge Economy. Princeton University Press.
""")